# 03 — Inference Demo

Runs fire & smoke detection on 3 demo videos.

---

In [ ]:
!pip install ultralytics gdown -q

from google.colab import drive
drive.mount('/content/drive')

import os, cv2, shutil
import numpy as np
import matplotlib.pyplot as plt
from base64 import b64encode
from IPython.display import display, HTML
from ultralytics import YOLO

# Download model
MODEL_ID = '1_EYUmnBE3a3PP0_f71VFBMESSsxv43m2'
BEST_PT  = '/content/best.pt'
if not os.path.exists(BEST_PT):
    !gdown --id {MODEL_ID} -O {BEST_PT}

# Download demo videos
DEMOS = {
    'demo1.mp4': '1o70TOThRv1uQQfc1GebNkOcLeq7Fk-EB',
    'demo2.mp4': '1pbCIQw2Sb1IcbqpSGBwu7Dv7N1-BYHyd',
    'demo3.mp4': '1xiLd1HnGgDXaVQfk0KJoMAmFI-yG--QJ',
}
DEMO_DIR   = '/content/demos'
OUTPUT_DIR = '/content/outputs'
DRIVE_OUTPUT = '/content/drive/MyDrive/AI_TRAINING/PCCC_yolo/demo_outputs'
os.makedirs(DEMO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

for name, fid in DEMOS.items():
    path = os.path.join(DEMO_DIR, name)
    if not os.path.exists(path):
        !gdown --id {fid} -O {path}

model = YOLO(BEST_PT)
CONF  = 0.25
print('Model loaded! Classes:', model.names)

## Run inference on all demo videos

In [ ]:
results_summary = {}

for name in DEMOS:
    video_path  = os.path.join(DEMO_DIR, name)
    output_path = os.path.join(OUTPUT_DIR, f'output_{name}')

    cap = cv2.VideoCapture(video_path)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f'\n{"="*50}')
    print(f'  {name} — {w}x{h} @ {fps:.1f}fps, {total} frames')
    print(f'{"="*50}')

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    frame_count = fire_frames = smoke_frames = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1
        result = model.predict(frame, conf=CONF, device=0, verbose=False)[0]
        detected = set(int(b.cls) for b in result.boxes)
        if 1 in detected: fire_frames += 1
        if 0 in detected: smoke_frames += 1
        out.write(result.plot())
        if frame_count % 200 == 0:
            print(f'  {frame_count}/{total} frames...')

    cap.release()
    out.release()

    results_summary[name] = {'total': frame_count, 'fire': fire_frames, 'smoke': smoke_frames, 'output': output_path}

    if frame_count > 0:
        print(f'  Fire  : {fire_frames}/{frame_count} frames ({fire_frames/frame_count*100:.1f}%)')
        print(f'  Smoke : {smoke_frames}/{frame_count} frames ({smoke_frames/frame_count*100:.1f}%)')

print(f'\n{"="*50}')
print('All videos processed!')

## Detection summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, info) in zip(axes, results_summary.items()):
    total = info['total']
    if total == 0:
        continue
    fire_pct  = info['fire']  / total * 100
    smoke_pct = info['smoke'] / total * 100
    clean_pct = max(0, 100 - max(fire_pct, smoke_pct))
    bars = ax.bar(['fire', 'smoke', 'clean'],
                  [fire_pct, smoke_pct, clean_pct],
                  color=['#ff5555', '#5599ff', '#88cc88'])
    ax.bar_label(bars, fmt='%.1f%%')
    ax.set_ylim(0, 110)
    ax.set_ylabel('% of frames')
    ax.set_title(name, fontweight='bold')

plt.suptitle('Detection Results per Video', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Sample frames

In [ ]:
for name in DEMOS:
    cap = cv2.VideoCapture(os.path.join(DEMO_DIR, name))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = [int(total * i / 5) for i in range(1, 5)]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, idx in zip(axes, indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            ax.axis('off')
            continue
        result = model.predict(frame, conf=CONF, device=0, verbose=False)[0]
        ax.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(f'frame {idx}', fontsize=9)
        ax.axis('off')

    cap.release()
    plt.suptitle(name, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Play output videos

In [ ]:
for name, info in results_summary.items():
    output_path = info['output']
    playable    = output_path.replace('.mp4', '_h264.mp4')
    size_mb     = os.path.getsize(output_path) / (1024*1024)

    print(f'\n--- {name} ({size_mb:.1f} MB) ---')

    # Re-encode to H.264 so browser can play it
    !ffmpeg -y -i {output_path} -vcodec libx264 -f mp4 {playable} -loglevel quiet

    if os.path.getsize(playable) / (1024*1024) > 200:
        print('Too large for inline playback.')
    else:
        b64 = b64encode(open(playable, 'rb').read()).decode()
        display(HTML(f'<h4>{name}</h4><video width="720" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## Save to Drive

Output videos saved to:
```
/content/drive/MyDrive/AI_TRAINING/PCCC_yolo/demo_outputs/
```

In [ ]:
for name, info in results_summary.items():
    dest = os.path.join(DRIVE_OUTPUT, f'output_{name}')
    shutil.copy2(info['output'], dest)
    print(f'Saved: {dest}')

print('\nAll outputs saved to Drive!')